# Lesson 06 — Query transformation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/06-query-transformation/colab.ipynb)

*Everything so far improved the index. This lesson improves the query — and asks whether it is worth a model call.*

**What you will learn**

- Why a user's question is not a good search query (vocabulary mismatch, framing)
- Four techniques — rewrite, HyDE, multi-query, step-back — and what each actually produces
- How several query variants are merged (RRF again, on a new axis)
- How HyDE fails: precisely and confidently wrong
- A measured negative result, and the general rule it teaches about fusion

| | |
|---|---|
| ⏱️ Time | 45–60 min (longer on CPU) |
| 🖥️ Runtime | **T4 GPU strongly recommended** — every query costs a model call |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [05-reranking](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/05-reranking/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
#@title ⚙️ Setup — run me first
USE_LLM = True  #@param {type:"boolean"}
BRANCH = "main"  #@param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH,
             "https://github.com/safar-byte/rag-from-scratch-to-production.git", str(REPO)],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup

LLM_READY = setup(llm=USE_LLM)

## 1. The idea

A user's question is an *input to* a retrieval system, not a search query. It carries
words that are useless for matching — *"can you tell me roughly what the default timeout
is"* contains one term that matters — and it often omits the words that would find the
answer. *"Why is it slow after a while?"* shares no vocabulary at all with a document
about unbounded cache growth.

That is the **vocabulary mismatch** problem, the oldest problem in information
retrieval. Query transformation attacks it by using an LLM to rewrite the query
*before* retrieval.

| Technique | What the model writes | Good for | Fails by |
|---|---|---|---|
| **Rewrite** | a short keyword query | chatty, conversational input | dropping the one rare term that mattered |
| **HyDE** | a *fake answer* passage, retrieved with | questions phrased far from the docs | hallucinating about the wrong subject |
| **Multi-query** | 3 alternative phrasings, fused | recall | cost — N retrievals (and N rerankings) |
| **Step-back** | a broader background question | "why" questions | adding noise to lookups |

Every one of them inserts **a full model call before retrieval starts**. That cost is
the real subject of this lesson.

In [ ]:
import time

import pandas as pd

from ragkit.pipeline import RagPipeline
from ragkit.query import TRANSFORMERS
from ragkit.query.transform import HYDE_SYSTEM, MULTI_QUERY_SYSTEM, REWRITE_SYSTEM, STEP_BACK_SYSTEM

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())

for name, prompt in [("REWRITE", REWRITE_SYSTEM), ("HYDE", HYDE_SYSTEM),
                     ("MULTI-QUERY", MULTI_QUERY_SYSTEM), ("STEP-BACK", STEP_BACK_SYSTEM)]:
    print(f"── {name} system prompt " + "─" * 40 + f"\n{prompt}\n")

## 2. See what each technique produces

🔮 **Predict:** for "Why do I get charged twice?" (the answer is about *prompt caching* —
paying repeatedly for the same tokens), which technique will produce a variant that
mentions caching?

In [ ]:
question = "Why do I get charged twice?"
rows = []
if LLM_READY:
    for name, cls in TRANSFORMERS.items():
        transformer = cls() if name == "identity" else cls(pipeline.generator)
        started = time.perf_counter()
        variants = transformer.transform(question)
        rows.append({"technique": name, "transform ms": round((time.perf_counter() - started) * 1000),
                     "variants (0 = original)": "\n".join(f"{i}. {v}" for i, v in enumerate(variants))})
    pd.set_option("display.max_colwidth", None)
    display(pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}))

Read the **transform ms** column against retrieval itself (tens of milliseconds once
the models are loaded, ~1s with reranking on CPU). On the author's CPU machine each
transform added ~25 *seconds*. That ratio, not the cleverness of the variants, decides
whether any of this ships.

Note that **variant 0 is always the original question**. Every technique keeps it,
because a transformation that drops the one rare term in the question is a common and
silent failure.

## 3. How the variants are merged

Several variants → several rankings → the same problem hybrid search solved. So
`TransformingRetriever` reuses **RRF**: "rankings that must be merged" does not care
whether the axis was different retrievers or different phrasings.

In [ ]:
print(open("ragkit/query/retriever.py").read().split("class TransformingRetriever")[1])

In [ ]:
if LLM_READY:
    pipeline.set_strategy("rerank", transform="multiquery")
    fused = pipeline.retrieve(question, top_k=5)
    cols = [c for c in results_frame(fused).columns if c.endswith("_rank") or c in ("doc_id", "rrf", "text")]
    display(results_frame(fused, chars=60)[cols])
    pipeline.set_strategy("rerank", transform="identity")

The `original_rank`, `q1_rank`, `q2_rank`… columns show where each phrasing ranked the
chunk. A chunk found by several phrasings gets several RRF votes.

## 4. HyDE, by hand

The premise of **H**ypothetical **D**ocument **E**mbeddings: an *answer* sits closer in
embedding space to the real answer than a *question* does, because answers resemble
answers. Let's test that premise directly — embed the question, embed a hypothetical
answer, and compare both to the passage that really answers it.

In [ ]:
import numpy as np

from ragkit.query import Hyde

embedder = pipeline.embedder
q = "How do I stop paying for the same tokens over and over again?"
target = next(c for c in pipeline._store.search(embedder.embed_query("prompt prefix cache"), top_k=10)
              if c.chunk.doc_id == "13-caching.md").chunk
print("target passage:", target.text[:160], "…\n")

if LLM_READY:
    fake = Hyde(pipeline.generator).transform(q)[-1]
    print("HyDE passage:", fake, "\n")
    t = np.array(embedder.embed_documents([target.text])[0])
    print(f"cosine(question,     target) = {np.array(embedder.embed_query(q)) @ t:.3f}")
    print(f"cosine(HyDE passage, target) = {np.array(embedder.embed_query(fake)) @ t:.3f}")

**Read the fake passage before the numbers.** The premise only pays when the fake
answer is actually *about the right thing* and uses the document's vocabulary. When
this notebook was written, the 1.5B model's "answer" was about account configuration,
not caching — and still scored a hundredth closer, which is noise, not evidence. A
larger model writes a passage about prompt caching and lands clearly closer. That is
the first lesson of HyDE: it is only as good as the model's knowledge of your domain.

Now push on that. Ask about something **specific to this corpus**, which a general
model cannot know:

In [ ]:
if LLM_READY:
    q2 = "What does exit status 75 from the indexer mean?"
    fake2 = Hyde(pipeline.generator).transform(q2)[-1]
    print("HyDE passage:", fake2, "\n")
    print("retrieved with the fake passage alone:")
    for s in pipeline.retrieve(fake2, top_k=3):
        print(f"   {s.chunk.doc_id:28s} {s.chunk.text[:70]!r}")
    print("\nthe truth (07-operations.md): exit status 75 means another indexer holds the lock.")

The model writes a confident, plausible passage about what exit status 75 means *in
general* — a generic error — when in this system it means a lock conflict. Here the fake
passage still contains "indexer" and "exit status 75", so retrieval may well survive;
the fake *answer* was wrong but its *vocabulary* was right.

The dangerous case is when the hallucination drifts in vocabulary too — the model
guesses a different subject entirely and retrieves confidently against it. **HyDE does
not degrade gracefully; when it fails, it fails by being precisely wrong.** That is why
`ragkit`'s `Hyde` retrieves with *both* the question and the fake passage instead of
discarding the question.

## 5. Measure it

`rewrite` over the reranked pipeline, against the same pipeline untouched, over all 44
questions. This makes 44 model calls — about a minute on a T4, several minutes on CPU.

🔮 **Predict** the sign of the change on `vocab R@5` — the category query rewriting
exists to help.

In [ ]:
from ragkit.eval.run import run_eval

RUN_OTHERS = False  #@param {type:"boolean"}   # hyde, multiquery, stepback: 3× more model calls

runs = {}
if LLM_READY:
    for transform in ["identity", "rewrite"] + (["hyde", "multiquery", "stepback"] if RUN_OTHERS else []):
        pipeline.set_strategy("rerank", transform=transform)
        started = time.perf_counter()
        runs[f"{transform} → rerank"] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
        print(f"{transform}: {time.perf_counter() - started:.0f}s")
    pipeline.set_strategy("rerank", transform="identity")
    display(compare_runs(runs).round(3))

### What the repo measured

| Pipeline | R@1 | R@3 | R@5 | nDCG@5 | vocab R@5 | ms |
|---|---|---|---|---|---|---|
| rerank (no transform) | **0.811** | **0.960** | **0.987** | **0.946** | **1.000** | **1613** |
| rewrite → rerank | 0.795 | 0.943 | 0.966 | 0.920 | 0.875 | 3507 |

**Worse on every metric, at 2.2× the latency** — including on the vocabulary-mismatch
questions, the category rewriting exists to help. Compare with your run above.

### Why — and why you have seen this before

The mechanism is lesson 04's. `Rewrite` keeps the original query *and* adds the
rewrite, then fuses the two rankings with RRF — **equally weighted**. When the rewrite
is worse than the original, equal-weight fusion drags a good ranking down with a bad
one. Same fusion, same failure, different axis: there it was two retrievers, here it is
two phrasings.

> **The general shape: adding a second opinion only helps if it is roughly as good as
> the first.** Fusion is not free insurance. It averages, and averaging with something
> worse makes things worse.

### Two structural reasons this corpus was never going to reward it

1. **Saturation.** Many questions already score 1.000 at every depth. A transformation
   cannot improve what is already perfect — it can only break it.
2. **The vocabulary-mismatch questions were already solved** by hybrid + reranking
   (vocab R@5 = 1.000 in lesson 05). Nothing was left for a rewrite to fix, and plenty
   for it to disturb.

Query transformation is a real technique for a real problem. On a corpus where that
problem has already been solved another way, it is a model call per query that buys
negative value. **That is the whole lesson, and you only get it by measuring.**

`hyde`, `multiquery` and `stepback` were never measured in the repo (each is ~20 min on
the author's CPU). Tick `RUN_OTHERS` on a GPU runtime and find out — the prior is that
they also lose, and multi-query should lose hardest since it fuses the most rankings. If
one of them wins, that is more interesting than anything written above.

## 6. Check your understanding

**1. Why does every transformer keep the original query as variant 0?**

<details><summary>▶ Show answer</summary>

Because a rewrite can silently drop the one rare, decisive term (an identifier, a
product name). Keeping the original costs one extra retrieval, not a model call, and
protects against that failure.

</details>

**2. Rewrite made retrieval *worse*. Explain the mechanism.**

<details><summary>▶ Show answer</summary>

The original and rewritten queries are fused with equal-weight RRF. When the
rewrite ranks worse than the original, its votes pull good results down — the same
way BM25 dragged dense down in lesson 04.

</details>

**3. When is HyDE most dangerous?**

<details><summary>▶ Show answer</summary>

When the question is about something specific to your corpus that the model does
not know (internal codes, product behaviour). The model writes a confident,
plausible, wrong passage and retrieval follows it precisely.

</details>

## ✍️ Exercises

1. **Route instead of applying.** Only transform queries that do *not* look like
   identifiers (`ragkit.retrieve.looks_lexical`, lesson 09). Does that recover the loss
   on `lookup` questions?
2. **Catch HyDE failing** on three more corpus-specific questions from the golden set
   (`kind == "lookup"`).
3. **Weight the original.** Change the fusion so the original query's ranking counts
   twice. Does rewrite stop losing? (And is that just lesson 04's weighted-RRF
   overfitting again?)

In [ ]:
# Exercise 1 — a starting point
from ragkit.query import Rewrite, TransformingRetriever
from ragkit.retrieve import looks_lexical


class RoutedRewrite:
    name = "routed-rewrite"

    def __init__(self, base, generator):
        self.base, self.wrapped = base, TransformingRetriever(base, Rewrite(generator))

    def retrieve(self, query, top_k=5, where=None):
        chosen = self.base if looks_lexical(query) else self.wrapped
        return chosen.retrieve(query, top_k=top_k, where=where)


if LLM_READY:
    pipeline.set_strategy("rerank")
    pipeline._retriever = RoutedRewrite(pipeline.retriever, pipeline.generator)
    runs["routed rewrite → rerank"] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
    pipeline.set_strategy("rerank")
    display(compare_runs(runs).round(3))

## Key takeaways

1. Query transformation fixes the query rather than the index, at the price of a model call before retrieval.
2. Variants are merged with RRF — and equal-weight fusion with a worse variant makes things worse.
3. HyDE fails by being precisely wrong on domain-specific questions.
4. On this corpus rewrite measured worse on every metric at 2.2× latency: a negative result, kept.
5. Ask "does it buy more than it costs *here*?" — and route expensive steps to the queries that need them.

## Next

**[Lesson 07 — Contextual retrieval](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/07-contextual-retrieval/colab.ipynb)**